# Hastaneye Yatış Süresi Tahminleme Modeli (Healthcare Length of Stay Prediction)

Bu notebook, **Microsoft Hospital Length of Stay** veri seti üzerinde Keşifsel Veri Analizi (EDA), Özellik Mühendisliği ve Makine Öğrenmesi modellemesi adımlarını içerir.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)

# Veri setini otomatik bul ve yükle
csv_files = glob.glob("../data/*.csv")
df = pd.read_csv(csv_files[0])
df.columns = [c.strip().lower() for c in df.columns]

print(f"Veri Seti Boyutu: {df.shape}")
df.head()

## 1. Veri Yapısı ve Eksik Değer Kontrolü

In [ ]:
df.info()
print("\nEksik Değer Sayıları:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 2. Keşifsel Veri Analizi (EDA) ve Görselleştirmeler

In [ ]:
os.makedirs("../reports/figures", exist_ok=True)

# 1. Hedef Değişken Dağılımı (Length of Stay)
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="lengthofstay", color="#2b5c8f")
plt.title("Hastanede Kalış Süresi (Gün) Dağılımı", fontsize=14, fontweight="bold")
plt.xlabel("Kalış Süresi (Gün)")
plt.ylabel("Hasta Sayısı")
plt.tight_layout()
plt.savefig("../reports/figures/los_distribution.png", dpi=300)
plt.show()

# 2. Tekrar Yatış Sayısı (rcount) vs Kalış Süresi
if "rcount" in df.columns:
    plt.figure(figsize=(10, 5))
    order = sorted(df["rcount"].astype(str).unique())
    sns.boxplot(data=df, x=df["rcount"].astype(str), y="lengthofstay", order=order)
    plt.title("Önceki Tekrar Yatış Sayısına Göre Kalış Süresi", fontsize=14, fontweight="bold")
    plt.xlabel("Son 180 Gündeki Tekrar Yatış Sayısı (rcount)")
    plt.ylabel("Kalış Süresi (Gün)")
    plt.tight_layout()
    plt.savefig("../reports/figures/rcount_vs_los.png", dpi=300)
    plt.show()

# 3. Korelasyon Isı Haritası
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(14, 10))
corr = numeric_df.corr()
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Sayısal Değişkenler Korelasyon Matrisi", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/correlation_heatmap.png", dpi=300)
plt.show()

## 3. Model Eğitimi ve Karşılaştırma
Modüler eğitim kodunu çalıştırarak Ridge, Random Forest ve XGBoost modellerini kıyaslayalım.

In [ ]:
import sys
sys.path.append("../src")
from data_preprocessing import prepare_train_test_data
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_train, X_test, y_train, y_test, preprocessor = prepare_train_test_data(csv_files[0])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(n_estimators=300, learning_rate=0.08, max_depth=6, random_state=42, n_jobs=-1))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R2:   {r2_score(y_test, y_pred):.4f}")